# CrazeAI — 1M-parameter Google Colab test and GGUF export

This isolated test notebook trains a **~951K-parameter** small decoder-only Transformer from scratch, exports it to Hugging Face format, converts it to GGUF, quantizes to `Q4_K_M`, enforces a strict **100,000,000-byte (100 MB)** limit, smoke-tests it, and downloads the result.

**The existing 135M workflow and its files are unchanged.** This notebook uses separate files under `training_jax_1m/`. Select **Runtime → Change runtime type → T4 GPU** if available. This is a pipeline test, not a capable conversational model.

In [ ]:
# Confirm runtime hardware before installing anything.
import os, sys, subprocess
print(sys.version)
try:
    import jax
    print('JAX:', jax.__version__)
    print('Backend:', jax.default_backend())
    print('Devices:', jax.devices())
except Exception as e:
    print('JAX import issue:', repr(e))
    raise
if jax.default_backend() != 'gpu':
    print('\nWARNING: JAX is not using a GPU. In Colab, select Runtime > Change runtime type > T4 GPU, reconnect, and rerun this cell.')

In [ ]:
# Clone the working branch and install dependencies without deliberately replacing Colab's JAX build.
%cd /content
!rm -rf CrazeAi llama.cpp
!git clone --depth 1 --branch test https://github.com/CrazeStudio/CrazeAi.git
%cd /content/CrazeAi
%pip install -q optax safetensors tokenizers datasets huggingface_hub
import jax, optax, numpy, tokenizers, datasets
print('Backend:', jax.default_backend(), '| Devices:', jax.devices())

In [ ]:
# Small first run: ~1M parameters, fewer examples, shorter context.
MAX_EXAMPLES = 200
EPOCHS = 1
SEQ_LEN = 64
BATCH_SIZE = 1
CHUNK_SIZE = 8
MAX_MODEL_BYTES = 100_000_000

import os
os.environ['PYTHONUNBUFFERED'] = '1'
!mkdir -p artifacts
!python training_jax/prepare_sft.py --max-examples {MAX_EXAMPLES} --output artifacts/sft.jsonl
!test -s artifacts/sft.jsonl && wc -l artifacts/sft.jsonl

In [ ]:
# Train the isolated ~951K-parameter model. This should be much quicker than 135M.
!python training_jax_1m/train.py --data artifacts/sft.jsonl --output artifacts/crazeai-1m-jax --epochs {EPOCHS} --seq-len {SEQ_LEN} --batch-size {BATCH_SIZE} --max-examples {MAX_EXAMPLES} --chunk-size {CHUNK_SIZE}
!test -s artifacts/crazeai-1m-jax/params.npz

In [ ]:
# Export and validate the small Hugging Face checkpoint.
!python training_jax_1m/export_hf.py --checkpoint artifacts/crazeai-1m-jax --output artifacts/crazeai-1m
!test -s artifacts/crazeai-1m/model.safetensors
!test -s artifacts/crazeai-1m/config.json
!ls -lh artifacts/crazeai-1m/

In [ ]:
# Build llama.cpp converter/quantizer.
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DGGML_NATIVE=OFF
!cmake --build llama.cpp/build --config Release -j2 --target llama-quantize llama-cli
%pip install -q -r /content/llama.cpp/requirements.txt

In [ ]:
# Convert to F16 GGUF, then quantize to Q4_K_M.
%cd /content/CrazeAi
!python /content/llama.cpp/convert_hf_to_gguf.py artifacts/crazeai-1m --outfile artifacts/crazeai-1m-f16.gguf --outtype f16
!test -s artifacts/crazeai-1m-f16.gguf
!/content/llama.cpp/build/bin/llama-quantize artifacts/crazeai-1m-f16.gguf artifacts/crazeai-1m-q4_k_m.gguf Q4_K_M
!test -s artifacts/crazeai-1m-q4_k_m.gguf
import os
model_path = '/content/CrazeAi/artifacts/crazeai-1m-q4_k_m.gguf'
size_bytes = os.path.getsize(model_path)
print(f'GGUF size: {size_bytes:,} bytes ({size_bytes / 1_000_000:.2f} MB)')
if size_bytes > MAX_MODEL_BYTES:
    raise RuntimeError(f'Model exceeds 100 MB: {size_bytes:,} bytes. Do not label this export as under 100 MB.')
print('PASS: final GGUF is within the strict 100 MB limit.')

In [ ]:
# Smoke test: generate a short response from the small quantized model.
! /content/llama.cpp/build/bin/llama-cli -m /content/CrazeAi/artifacts/crazeai-1m-q4_k_m.gguf -p 'Who are you?' -n 32 --temp 0.2 --no-display-prompt

In [ ]:
# Download the small test model to your device.
from google.colab import files
files.download('/content/CrazeAi/artifacts/crazeai-1m-q4_k_m.gguf')

## Notes
- The model has about **951,000 parameters** (vocabulary 2,048; hidden size 128; 2 layers; 4 attention heads; intermediate size 384). It is randomly initialized and trained on only a small sample, so answers will be poor; the goal is pipeline validation.
- The original 135M implementation and GitHub Actions workflow are not modified by this notebook change.
- The output-size cell fails if the final GGUF exceeds 100,000,000 bytes.
- If Colab disconnects or runs out of memory, rerun in a fresh session and reduce `MAX_EXAMPLES`; save artifacts to Google Drive if you need to preserve checkpoints.
- Use only data you are permitted to train on.